# Milestone 4 — Formulating MCQ Task & Fine-Tuning with LoRA

**Project:** Smart MCQ Solver Challenge  
**Model:** microsoft/deberta-v3-base  
**Technique:** LoRA (Low-Rank Adaptation) — parameter-efficient fine-tuning  
**Task Formulation:** Multiple Choice Classification  
**Metric:** MAP@3 (Mean Average Precision at 3)

---

### Why Fine-Tuning Changes Everything

In Milestones 2 and 3, we used pretrained models **as-is** — they were never shown our MCQ data. We relied on general-purpose similarity or entailment to rank options. This has a fundamental ceiling because:

- The model doesn't know what "correct answer" means in our specific context
- It can't learn patterns like "options that paraphrase the prompt are often distractors"
- It has no feedback mechanism — it never learns from its mistakes

**Fine-tuning solves all of this.** We take a powerful pretrained model and train it **directly on our MCQ data** with supervision — it sees the question, all 5 options, and which one is correct. Over thousands of examples, it learns to distinguish correct answers from plausible distractors.

### Why LoRA Instead of Full Fine-Tuning

DeBERTa-v3-base has ~86 million parameters. Full fine-tuning would:
- Require enormous GPU memory to store gradients for all 86M parameters
- Risk catastrophic forgetting — overwriting the useful knowledge the model already has
- Need large datasets to avoid overfitting (we only have 2000 samples)

**LoRA (Low-Rank Adaptation)** elegantly solves this:
- Freezes ALL 86M original parameters
- Adds tiny trainable "adapter" matrices to attention layers
- Only trains ~0.5M parameters (less than 1% of the model)
- Preserves the model's pretrained knowledge while learning our task
- Fits comfortably in Kaggle's GPU memory

### Why DeBERTa-v3 Over BERT/RoBERTa

DeBERTa-v3 uses **disentangled attention** — it separately encodes content and position information, then combines them with a disentangled attention mechanism. This gives it:
- Better understanding of word relationships and ordering
- Superior performance on NLU benchmarks compared to BERT and RoBERTa
- An enhanced mask decoder that makes it better at discriminating between options

**Note:** This notebook requires GPU. On Kaggle: Settings → Accelerator → GPU T4 x2

In [1]:
# Install compatible versions
# peft==0.13.0 avoids the torchao compatibility issue on Kaggle
!pip install peft==0.13.0 accelerate bitsandbytes wandb -q

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
import wandb
import gc
import os
import warnings
warnings.filterwarnings('ignore')

# Verify GPU availability — this notebook REQUIRES GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")
else:
    print("WARNING: No GPU detected. Fine-tuning will be extremely slow.")
    print("Enable GPU in Kaggle: Settings → Accelerator → GPU T4 x2")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.5/322.5 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 26.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.

## 1. Load Dataset & Define Evaluation Utilities

We load the same competition dataset and set up:
- **Label mapping** — convert A/B/C/D/E to numeric indices 0–4
- **MAP@3 functions** — our competition metric for evaluation
- **Detailed breakdown** — shows where the model succeeds and fails by position

In [2]:
# Load competition data
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

option_cols = ['A', 'B', 'C', 'D', 'E']

# Label encoding: A=0, B=1, C=2, D=3, E=4
label_to_idx = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
idx_to_label = {v: k for k, v in label_to_idx.items()}

print(f"Train: {train_df.shape} ({train_df.shape[0]} questions)")
print(f"Test:  {test_df.shape} ({test_df.shape[0]} questions)")
print(f"\nAnswer distribution:\n{train_df['answer'].value_counts().sort_index()}")


# ── MAP@3 Evaluation Functions ──────────────────────────────────

def ap_at_3(true_label, predicted_labels):
    """Score a single question: 1/position if correct in top 3, else 0."""
    for i, pred in enumerate(predicted_labels[:3]):
        if pred.strip().upper() == true_label.strip().upper():
            return 1.0 / (i + 1)
    return 0.0

def map_at_3(true_labels, predicted_labels):
    """Mean AP@3 across all questions."""
    return np.mean([ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)])

def map_at_3_detailed(true_labels, predicted_labels):
    """MAP@3 with full position-level breakdown."""
    scores = [ap_at_3(t, p) for t, p in zip(true_labels, predicted_labels)]
    n = len(scores)
    return {
        'map3': np.mean(scores),
        'correct_at_1': sum(1 for s in scores if s == 1.0),
        'correct_at_2': sum(1 for s in scores if s == 0.5),
        'correct_at_3': sum(1 for s in scores if abs(s - 1/3) < 0.01),
        'missed': sum(1 for s in scores if s == 0.0),
        'total': n,
        'top1_acc': sum(1 for s in scores if s == 1.0) / n,
        'top3_acc': sum(1 for s in scores if s > 0) / n,
    }

def print_results(name, results):
    """Pretty-print evaluation results."""
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(f"  MAP@3:          {results['map3']:.4f}")
    print(f"  Top-1 Accuracy: {results['top1_acc']:.2%}")
    print(f"  Top-3 Accuracy: {results['top3_acc']:.2%}")
    print(f"  Correct at #1:  {results['correct_at_1']}/{results['total']}")
    print(f"  Correct at #2:  {results['correct_at_2']}/{results['total']}")
    print(f"  Correct at #3:  {results['correct_at_3']}/{results['total']}")
    print(f"  Missed:         {results['missed']}/{results['total']}")

print("\nSetup complete.")

Train: (2000, 8) (2000 questions)
Test:  (500, 7) (500 questions)

Answer distribution:
answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64

Setup complete.


## 2. W&B Login

Login to Weights & Biases for experiment tracking. API key is stored as a Kaggle Secret named `WANDB_API_KEY`.

In [3]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
wandb.login(key=secrets.get_secret("WANDB_API_KEY"))
print("W&B login successful!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


W&B login successful!


## 3. Train/Validation Split

We split the training data **85% train / 15% validation** to:
- Monitor overfitting during training (if val loss goes up while train loss goes down → overfitting)
- Enable early stopping (stop training when val performance plateaus)
- Select the best checkpoint (the epoch with highest val MAP@3)

We use **stratified splitting** to ensure the answer distribution (A/B/C/D/E proportions) is preserved in both sets. Without stratification, random chance could give us a validation set with very few "E" answers, making evaluation unreliable.

In [4]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.15,         # 15% for validation
    random_state=42,        # reproducibility
    stratify=train_df['answer']  # preserve answer distribution
)

# Reset indices so iloc[0] works correctly later
train_data = train_data.reset_index(drop=True)
val_data = val_data.reset_index(drop=True)

print(f"Training set:   {len(train_data)} samples")
print(f"Validation set: {len(val_data)} samples")
print(f"\nTrain answer distribution:\n{train_data['answer'].value_counts().sort_index()}")
print(f"\nVal answer distribution:\n{val_data['answer'].value_counts().sort_index()}")

# Verify stratification preserved proportions
print(f"\nTrain 'B' proportion: {(train_data['answer'] == 'B').mean():.3f}")
print(f"Val 'B' proportion:   {(val_data['answer'] == 'B').mean():.3f}")
print(f"Full 'B' proportion:  {(train_df['answer'] == 'B').mean():.3f}")

Training set:   1700 samples
Validation set: 300 samples

Train answer distribution:
answer
A    314
B    417
C    390
D    304
E    275
Name: count, dtype: int64

Val answer distribution:
answer
A    55
B    73
C    69
D    54
E    49
Name: count, dtype: int64

Train 'B' proportion: 0.245
Val 'B' proportion:   0.243
Full 'B' proportion:  0.245


## 4. Data Formatting for Multiple Choice

### How `AutoModelForMultipleChoice` Expects Data

This is the most important conceptual step. The model doesn't see "a question with 5 options." Instead, each question becomes **5 independent text pairs**:

All 5 pairs pass through a **shared encoder** (DeBERTa). The encoder produces a representation for each pair, and a **classification head** on top outputs one logit per pair. The option with the highest logit is the predicted answer.

### Tensor Shapes

- **Input:** `(batch_size, 5, max_length)` — 5 sequences per question
- **Output logits:** `(batch_size, 5)` — one score per option
- **Labels:** `(batch_size,)` — index of the correct option (0–4)

### Why `max_length=256`

Most prompts are 50–150 characters, and most options are 50–200 characters. After tokenization, most prompt+option pairs fit within 256 tokens. Going longer (384, 512) captures more text but uses proportionally more GPU memory — we start at 256 and can increase if needed.

In [5]:
# ── Model & Tokenizer ──────────────────────────────────────────

MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")


# ── Custom Dataset Class ───────────────────────────────────────

class MCQDataset(Dataset):
    """
    Formats MCQ data for AutoModelForMultipleChoice.
    
    Each __getitem__ returns:
        input_ids:      (5, max_length) — tokenized (prompt, option) pairs
        attention_mask: (5, max_length) — 1 for real tokens, 0 for padding
        labels:         scalar — index of correct option (0-4)
    """
    
    def __init__(self, df, tokenizer, max_length=256, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.is_test = is_test
        self.option_cols = ['A', 'B', 'C', 'D', 'E']
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        # Create 5 (prompt, option) pairs
        # The tokenizer handles [CLS], [SEP] insertion automatically
        first_sentences = [prompt] * 5
        second_sentences = [str(row[col]) for col in self.option_cols]
        
        tokenized = self.tokenizer(
            first_sentences,          # 5 copies of the prompt
            second_sentences,         # 5 different options
            truncation=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        
        # Build output dictionary
        item = {
            'input_ids': tokenized['input_ids'],            # (5, max_length)
            'attention_mask': tokenized['attention_mask'],    # (5, max_length)
        }
        
        # DeBERTa-v3 may or may not use token_type_ids
        if 'token_type_ids' in tokenized:
            item['token_type_ids'] = tokenized['token_type_ids']
        
        # Add label only for train/val (not test)
        if not self.is_test:
            label = label_to_idx[row['answer']]
            item['labels'] = torch.tensor(label, dtype=torch.long)
        
        return item


# ── Create All Three Datasets ──────────────────────────────────

train_dataset = MCQDataset(train_data, tokenizer, MAX_LENGTH)
val_dataset = MCQDataset(val_data, tokenizer, MAX_LENGTH)
test_dataset = MCQDataset(test_df, tokenizer, MAX_LENGTH, is_test=True)

# Verify shapes by inspecting first sample
sample = train_dataset[0]
print(f"\nDataset sample shapes:")
print(f"  input_ids:      {sample['input_ids'].shape}")        # → (5, 256)
print(f"  attention_mask:  {sample['attention_mask'].shape}")   # → (5, 256)
print(f"  label:           {sample['labels'].item()} ({idx_to_label[sample['labels'].item()]})")

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)}")
print(f"  Val:   {len(val_dataset)}")
print(f"  Test:  {len(test_dataset)}")

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

Tokenizer loaded: microsoft/deberta-v3-base
Vocab size: 128000

Dataset sample shapes:
  input_ids:      torch.Size([5, 256])
  attention_mask:  torch.Size([5, 256])
  label:           1 (B)

Dataset sizes:
  Train: 1700
  Val:   300
  Test:  500


## 5. Load DeBERTa-v3-base with LoRA Adapters

### How LoRA Works Internally

In a standard transformer attention layer, the model computes:

Where `W_q`, `W_k`, `W_v` are large weight matrices (768×768 for base models).

**LoRA modifies this** by adding a low-rank decomposition:

Q = X × W_q + X × B_q × A_q (original + adapter)


Where `B_q` is (768 × r) and `A_q` is (r × 768). With r=16, this means:
- Original `W_q`: 768 × 768 = 589,824 parameters (FROZEN)
- LoRA adapter: 768 × 16 + 16 × 768 = 24,576 parameters (TRAINABLE)
- That's only **4.2%** of the original layer's parameters

### Our LoRA Configuration

| Parameter | Value | Why |
|-----------|-------|-----|
| `r=16` | Rank of decomposition | Good balance of capacity vs efficiency. r=8 is minimal, r=32 is heavy |
| `lora_alpha=32` | Scaling factor | Typically 2× the rank. Controls how much the adapter influences output |
| `lora_dropout=0.1` | Dropout on adapters | Prevents overfitting on our small dataset (2000 samples) |
| `target_modules` | query & value projections | These are the most impactful attention components for classification |
| `bias="none"` | Don't train biases | Reduces parameters further without hurting performance |

In [6]:
# Load the base multiple-choice model
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

total_base_params = sum(p.numel() for p in model.parameters())
print(f"Base model: {MODEL_NAME}")
print(f"Total base parameters: {total_base_params:,}")

# First, let's inspect the model's layer names to find the correct target modules
# DeBERTa-v3 uses different naming than BERT
print(f"\nSearching for projection layer names...")
target_candidates = []
for name, module in model.named_modules():
    if 'proj' in name.lower() or 'query' in name.lower() or 'value' in name.lower():
        if hasattr(module, 'weight'):
            target_candidates.append(name)

# Show first few to identify the naming pattern
for name in target_candidates[:10]:
    print(f"  Found: {name}")

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
classifier.weight                       | MISSING    | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight              

Base model: microsoft/deberta-v3-base
Total base parameters: 184,422,913

Searching for projection layer names...
  Found: deberta.encoder.layer.0.attention.self.query_proj
  Found: deberta.encoder.layer.0.attention.self.key_proj
  Found: deberta.encoder.layer.0.attention.self.value_proj
  Found: deberta.encoder.layer.1.attention.self.query_proj
  Found: deberta.encoder.layer.1.attention.self.key_proj
  Found: deberta.encoder.layer.1.attention.self.value_proj
  Found: deberta.encoder.layer.2.attention.self.query_proj
  Found: deberta.encoder.layer.2.attention.self.key_proj
  Found: deberta.encoder.layer.2.attention.self.value_proj
  Found: deberta.encoder.layer.3.attention.self.query_proj


model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

### 5.1 Apply LoRA Configuration

Based on the layer names discovered above, we configure LoRA to target the query and value projection layers in every attention block.

In [7]:
# Configure LoRA
# target_modules will be matched against layer names — DeBERTa-v3 typically uses
# "query_proj" and "value_proj" (unlike BERT which uses "query" and "value")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,       # sequence classification (closest to multiple choice)
    r=16,                              # rank of decomposition
    lora_alpha=32,                     # scaling factor (2x rank is standard)
    lora_dropout=0.1,                  # dropout for regularization
    target_modules=["query_proj", "value_proj"],  # which layers get adapters
    bias="none",                       # don't train bias terms
)

# Apply LoRA — this freezes the base model and adds adapter layers
model = get_peft_model(model, lora_config)

# Print parameter breakdown
model.print_trainable_parameters()

# Manual verification
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)
print(f"\n  Trainable:  {trainable:,} parameters")
print(f"  Frozen:     {frozen:,} parameters")
print(f"  Ratio:      {100 * trainable / (trainable + frozen):.2f}% trainable")

# Move model to GPU
model = model.to(device)
print(f"\nModel moved to {device}")

trainable params: 590,593 || all params: 185,013,506 || trainable%: 0.3192

  Trainable:  590,593 parameters
  Frozen:     184,422,913 parameters
  Ratio:      0.32% trainable

Model moved to cuda


## 6. Data Collator

The HuggingFace `Trainer` expects a **data collator** that takes a list of individual samples and stacks them into a batch. Our custom collator:

1. Stacks `input_ids` from each sample: list of (5, 256) tensors → (batch_size, 5, 256)
2. Stacks `attention_mask` similarly
3. Stacks labels: list of scalars → (batch_size,)

This is necessary because `AutoModelForMultipleChoice` expects the batch dimension as the first axis, with the 5 choices as the second axis.

In [8]:
class MCQDataCollator:
    """
    Batches multiple-choice samples into the format expected by
    AutoModelForMultipleChoice: (batch_size, num_choices, seq_length)
    """
    def __call__(self, features):
        batch = {
            'input_ids': torch.stack([f['input_ids'] for f in features]),
            'attention_mask': torch.stack([f['attention_mask'] for f in features]),
        }
        
        # Include token_type_ids only if present
        if 'token_type_ids' in features[0]:
            batch['token_type_ids'] = torch.stack([f['token_type_ids'] for f in features])
        
        # Include labels only if present (not for test set)
        if 'labels' in features[0]:
            batch['labels'] = torch.stack([f['labels'] for f in features])
        
        return batch

data_collator = MCQDataCollator()

# Verify the collator works correctly
test_batch = data_collator([train_dataset[0], train_dataset[1]])
print("Batch shapes after collation:")
for key, val in test_batch.items():
    print(f"  {key}: {val.shape}")
# Expected: input_ids (2, 5, 256), attention_mask (2, 5, 256), labels (2,)

Batch shapes after collation:
  input_ids: torch.Size([2, 5, 256])
  attention_mask: torch.Size([2, 5, 256])
  token_type_ids: torch.Size([2, 5, 256])
  labels: torch.Size([2])


## 7. Evaluation Metrics for Trainer

During training, the `Trainer` calls our `compute_metrics` function at the end of every epoch. It receives:
- `logits`: shape (N, 5) — raw model scores for each option
- `labels`: shape (N,) — correct option index

We compute:
- **MAP@3** — our competition metric, used for selecting the best checkpoint
- **Top-1 Accuracy** — how often the highest-scored option is correct
- **Top-3 Accuracy** — how often the correct option is in the top 3

The model logits are NOT probabilities — they're raw scores. We rank them directly (higher = more confident). For probabilities, we'd apply softmax, but for ranking purposes raw logits work identically.

In [9]:
def compute_metrics(eval_pred):
    """
    Called by Trainer at each evaluation step.
    Converts logits to top-3 predictions and computes MAP@3.
    """
    logits, labels = eval_pred  # logits: (N, 5), labels: (N,)
    
    # Top-1 accuracy: is the highest logit the correct answer?
    preds_top1 = np.argmax(logits, axis=1)
    accuracy = (preds_top1 == labels).mean()
    
    # MAP@3: rank all 5 options, take top 3
    predicted_labels = []
    true_labels = []
    
    for i in range(len(labels)):
        # argsort descending → indices of options ranked by score
        top3_indices = np.argsort(logits[i])[::-1][:3]
        top3_labels = [idx_to_label[idx] for idx in top3_indices]
        predicted_labels.append(top3_labels)
        true_labels.append(idx_to_label[labels[i]])
    
    map3 = map_at_3(true_labels, predicted_labels)
    
    # Top-3 accuracy
    top3_acc = sum(
        1 for i in range(len(labels))
        if labels[i] in np.argsort(logits[i])[::-1][:3]
    ) / len(labels)
    
    return {
        'accuracy': accuracy,
        'map3': map3,
        'top3_accuracy': top3_acc,
    }

print("Metrics function defined.")

Metrics function defined.


## 8. Training Configuration

### Hyperparameter Choices Explained

| Parameter | Value | Reasoning |
|-----------|-------|-----------|
| **Learning rate** | 2e-5 | Standard for transformer fine-tuning. Higher (5e-5) risks instability, lower (1e-5) converges too slowly |
| **Batch size** | 4 | Each sample has 5 option pairs × 256 tokens — larger batches exceed T4 memory |
| **Gradient accumulation** | 4 | Effective batch size = 4 × 4 = 16. Simulates larger batch without the memory cost |
| **Epochs** | 10 | We train up to 10 but early stopping will likely halt around epoch 3–6 |
| **Early stopping patience** | 2 | Stop if MAP@3 doesn't improve for 2 consecutive epochs — prevents overfitting |
| **fp16** | True | Mixed precision halves memory for most operations with negligible accuracy impact |
| **Warmup ratio** | 0.1 | First 10% of steps use a linearly increasing learning rate to prevent early instability |
| **Weight decay** | 0.01 | L2 regularization — gently penalizes large weights to reduce overfitting |
| **metric_for_best_model** | map3 | We save the checkpoint with the highest validation MAP@3, not the lowest loss |

### What Happens During Each Training Step

1. A batch of 4 questions (each with 5 options) is loaded
2. Forward pass: model scores all 20 prompt-option pairs
3. Cross-entropy loss is computed (correct option should have highest logit)
4. Backward pass: gradients flow only through LoRA adapter parameters (base model is frozen)
5. Every 4 steps: optimizer updates weights (gradient accumulation)
6. Every epoch: evaluate on validation set, log metrics to W&B, save checkpoint if best

In [10]:
PROJECT_NAME = "22f3002548-t22026"

training_args = TrainingArguments(
    output_dir='./mcq_finetuned',
    
    # ── Core Training ──
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,      # can use larger batch for eval (no gradients)
    gradient_accumulation_steps=4,      # effective batch = 4 × 4 = 16
    
    # ── Optimizer ──
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,                   # warm up for first 10% of steps
    
    # ── Memory Optimization ──
    fp16=True,                          # mixed precision training
    dataloader_num_workers=2,
    
    # ── Evaluation & Checkpointing ──
    eval_strategy="epoch",              # evaluate after every epoch
    save_strategy="epoch",              # save checkpoint after every epoch
    logging_steps=10,                   # log training loss every 10 steps
    load_best_model_at_end=True,        # load best checkpoint when training ends
    metric_for_best_model="map3",       # select best by MAP@3
    greater_is_better=True,             # higher MAP@3 = better
    save_total_limit=2,                 # keep only 2 best checkpoints (saves disk)
    
    # ── W&B Logging ──
    report_to="wandb",
    run_name="m4-deberta-v3-lora-mcq",
)

# Print key configuration
effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
steps_per_epoch = len(train_dataset) // (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps)
total_steps = steps_per_epoch * training_args.num_train_epochs

print(f"Training Configuration:")
print(f"  Model:              {MODEL_NAME}")
print(f"  Max length:         {MAX_LENGTH} tokens")
print(f"  Effective batch:    {effective_batch}")
print(f"  Steps per epoch:    {steps_per_epoch}")
print(f"  Max total steps:    {total_steps}")
print(f"  Learning rate:      {training_args.learning_rate}")
print(f"  Early stopping:     patience=2 on MAP@3")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training Configuration:
  Model:              microsoft/deberta-v3-base
  Max length:         256 tokens
  Effective batch:    16
  Steps per epoch:    106
  Max total steps:    1060
  Learning rate:      2e-05
  Early stopping:     patience=2 on MAP@3


## 9. Train the Model

This is where the actual learning happens. The Trainer handles:
- Forward pass, loss computation, backward pass, optimizer step
- Gradient accumulation and mixed precision automatically
- Evaluation on validation set after each epoch
- Saving checkpoints and selecting the best one
- Logging all metrics to W&B in real-time

**What to watch for in the training output:**
- `train_loss` should decrease steadily each epoch
- `eval_map3` should increase — this is what we care about
- If `eval_loss` starts increasing while `train_loss` keeps decreasing → overfitting (early stopping will catch this)
- Training typically takes **15–30 minutes** on T4 GPU

**Expected performance:**
- Epoch 1: MAP@3 ~0.40–0.55 (model is starting to learn)
- Epoch 3–5: MAP@3 ~0.60–0.75 (model has learned the task)
- Epoch 5+: diminishing returns, risk of overfitting

In [11]:
PROJECT_NAME = "22f3002548-t22026"

training_args = TrainingArguments(
    output_dir='./mcq_finetuned',
    
    # ── Core Training ──
    num_train_epochs=10,
    per_device_train_batch_size=2,      # reduced from 4 to save memory without fp16
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=8,      # effective batch = 2 × 8 = 16
    
    # ── Optimizer ──
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    
    # ── Memory (fp16 REMOVED — DeBERTa-v3 is incompatible) ──
    fp16=False,
    dataloader_num_workers=2,
    
    # ── Evaluation & Checkpointing ──
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="map3",
    greater_is_better=True,
    save_total_limit=2,
    
    # ── W&B Logging ──
    report_to="wandb",
    run_name="m4-deberta-v3-lora-mcq",
)

effective_batch = training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps
print(f"Training Configuration:")
print(f"  Batch size:         {training_args.per_device_train_batch_size}")
print(f"  Gradient accum:     {training_args.gradient_accumulation_steps}")
print(f"  Effective batch:    {effective_batch}")
print(f"  fp16:               DISABLED (DeBERTa-v3 incompatible)")
print(f"  Learning rate:      {training_args.learning_rate}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Training Configuration:
  Batch size:         2
  Gradient accum:     8
  Effective batch:    16
  fp16:               DISABLED (DeBERTa-v3 incompatible)
  Learning rate:      2e-05


## 9. Train the Model

This is where the actual learning happens. The Trainer handles:
- Forward pass, loss computation, backward pass, optimizer step
- Gradient accumulation and mixed precision automatically
- Evaluation on validation set after each epoch
- Saving checkpoints and selecting the best one
- Logging all metrics to W&B in real-time

**What to watch for in the training output:**
- `train_loss` should decrease steadily each epoch
- `eval_map3` should increase — this is what we care about
- If `eval_loss` starts increasing while `train_loss` keeps decreasing → overfitting (early stopping will catch this)
- Training typically takes **15–30 minutes** on T4 GPU

**Expected performance:**
- Epoch 1: MAP@3 ~0.40–0.55 (model is starting to learn)
- Epoch 3–5: MAP@3 ~0.60–0.75 (model has learned the task)
- Epoch 5+: diminishing returns, risk of overfitting

In [12]:
# Initialize W&B run with full configuration logging
wandb.init(
    project=PROJECT_NAME,
    name="m4-deberta-v3-lora-mcq",
    tags=["milestone4", "finetuning", "lora", "deberta-v3"],
    config={
        "model": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.1,
        "target_modules": "query_proj, value_proj",
        "learning_rate": training_args.learning_rate,
        "epochs": training_args.num_train_epochs,
        "effective_batch_size": effective_batch,
        "train_samples": len(train_dataset),
        "val_samples": len(val_dataset),
        "early_stopping_patience": 2,
    }
)

# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

# Start training
print("Starting fine-tuning...")
print(f"Training {sum(p.numel() for p in model.parameters() if p.requires_grad):,} parameters")
print(f"Base model ({sum(p.numel() for p in model.parameters() if not p.requires_grad):,} parameters) is FROZEN\n")

train_result = trainer.train()

# Print training summary
print(f"\n{'='*55}")
print(f"  Training Complete!")
print(f"{'='*55}")
print(f"  Total time:       {train_result.metrics['train_runtime']:.0f} seconds ({train_result.metrics['train_runtime']/60:.1f} minutes)")
print(f"  Final train loss: {train_result.metrics['train_loss']:.4f}")
print(f"  Epochs completed: {int(train_result.metrics.get('epoch', training_args.num_train_epochs))}")

wandb: Tracking run with wandb version 0.25.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260721_160812-yyzygigy
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run m4-deberta-v3-lora-mcq
wandb: ⭐️ View project at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026
wandb: 🚀 View run at https://wandb.ai/22f3002548-dl-genai-project/22f3002548-t22026/runs/yyzygigy


Starting fine-tuning...
Training 590,593 parameters
Base model (184,422,913 parameters) is FROZEN



Epoch,Training Loss,Validation Loss,Accuracy,Map3,Top3 Accuracy
1,25.747070,3.216797,0.203333,0.380556,0.623333
2,25.831445,3.210938,0.230000,0.392778,0.616667
3,25.760547,3.201172,0.240000,0.397778,0.613333
4,25.397852,3.187500,0.240000,0.424444,0.680000
5,23.345898,3.179688,0.273333,0.450556,0.703333
6,25.585742,3.171875,0.263333,0.447778,0.703333
7,25.501562,3.162109,0.276667,0.461111,0.720000
8,25.482617,3.154297,0.293333,0.472222,0.723333
9,25.427148,3.152344,0.303333,0.479444,0.720000
10,23.053320,3.150391,0.296667,0.473889,0.720000



  Training Complete!
  Total time:       1219 seconds (20.3 minutes)
  Final train loss: 25.2113
  Epochs completed: 10


## 10. Evaluate on Validation Set

After training, we run a final evaluation on the validation set using the **best checkpoint** (the one with highest MAP@3 during training, automatically loaded by `load_best_model_at_end=True`).

This tells us how well the model generalizes to questions it hasn't seen during training.

In [13]:
# Evaluate best checkpoint on validation set
eval_results = trainer.evaluate()

print(f"Validation Results (best checkpoint):")
print(f"  Loss:           {eval_results['eval_loss']:.4f}")
print(f"  MAP@3:          {eval_results['eval_map3']:.4f}")
print(f"  Top-1 Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  Top-3 Accuracy: {eval_results['eval_top3_accuracy']:.4f}")

Validation Results (best checkpoint):
  Loss:           3.1523
  MAP@3:          0.4794
  Top-1 Accuracy: 0.3033
  Top-3 Accuracy: 0.7200


## 11. Evaluate on Full Training Set

To compare fairly with Milestones 1–3 (where we evaluated on the entire training set), we also run inference on all 2000 training examples.

**Note:** This includes the 1700 samples the model trained on, so the score will be higher than the validation score. The validation score is the honest measure of generalization. We compute this full-train score only for comparison with previous milestones.

In [14]:
# Create dataset from the full training set (all 2000 rows)
full_train_dataset = MCQDataset(train_df, tokenizer, MAX_LENGTH)

# Run inference
print("Running inference on full training set (2000 questions)...")
full_predictions = trainer.predict(full_train_dataset)
full_logits = full_predictions.predictions  # shape: (2000, 5)

# Convert logits → top-3 label predictions
train_pred_labels = []
for i in range(len(full_logits)):
    top3_indices = np.argsort(full_logits[i])[::-1][:3]
    top3_labels = [idx_to_label[idx] for idx in top3_indices]
    train_pred_labels.append(top3_labels)

# Compute detailed metrics
results_ft = map_at_3_detailed(train_df['answer'].tolist(), train_pred_labels)
print_results("Fine-tuned DeBERTa-v3 (LoRA) — Full Train Set", results_ft)

# Log to W&B
wandb.log({
    "full_train_map3": results_ft['map3'],
    "full_train_top1_acc": results_ft['top1_acc'],
    "full_train_top3_acc": results_ft['top3_acc'],
    "full_train_missed": results_ft['missed'],
})

Running inference on full training set (2000 questions)...

  Fine-tuned DeBERTa-v3 (LoRA) — Full Train Set
  MAP@3:          0.4867
  Top-1 Accuracy: 31.55%
  Top-3 Accuracy: 72.15%
  Correct at #1:  631/2000
  Correct at #2:  430/2000
  Correct at #3:  382/2000
  Missed:         557/2000


## 12. Generate Kaggle Submission

We run the fine-tuned model on the test set (500 questions) and create the submission file. The model outputs 5 logits per question — we sort them descending and take the top 3 option labels.

In [15]:
# Run inference on test set
print("Running inference on test set (500 questions)...")
test_predictions = trainer.predict(test_dataset)
test_logits = test_predictions.predictions  # shape: (500, 5)

# Convert logits to top-3 label predictions
test_pred_labels = []
for i in range(len(test_logits)):
    top3_indices = np.argsort(test_logits[i])[::-1][:3]
    top3_labels = [idx_to_label[idx] for idx in top3_indices]
    test_pred_labels.append(top3_labels)

# Create submission DataFrame
submission = pd.DataFrame({
    'id': test_df['id'],
    'prediction': [' '.join(pred) for pred in test_pred_labels]
})

print(f"Submission shape: {submission.shape}")
print(f"\nSample predictions:")
print(submission.head(10))

# Save
submission.to_csv('submission.csv', index=False)
print(f"\nSaved to submission.csv")

Running inference on test set (500 questions)...


Submission shape: (500, 2)

Sample predictions:
   id prediction
0   1      B D E
1   2      D B C
2   3      A C D
3   4      C E A
4   5      E D A
5   6      B C D
6   7      D E C
7   8      A E B
8   9      D C E
9  10      E D B

Saved to submission.csv


## 13. Save LoRA Adapter & Logits

We save two things for future use:

**1. LoRA adapter weights (~2MB)** — just the small trained adapter matrices, not the full model. These can be loaded on top of the base DeBERTa model at any time to recreate our fine-tuned model. Useful for:
- Uploading to Kaggle Datasets for a separate inference notebook
- Deploying on HuggingFace Hub (bonus marks)
- Loading in the ensemble notebook (Milestone 5)

**2. Raw logits (numpy arrays)** — the model's scores for every option on every question. These are essential for Milestone 5 ensembling, where we'll combine logits from multiple models using weighted averaging.

In [16]:
# Save the LoRA adapter (NOT the full model — just the trained adapter weights)
adapter_path = "./lora_adapter_deberta_v3"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

# Check saved file sizes
total_size = sum(
    os.path.getsize(os.path.join(adapter_path, f))
    for f in os.listdir(adapter_path)
    if os.path.isfile(os.path.join(adapter_path, f))
)
print(f"LoRA adapter saved to: {adapter_path}")
print(f"Total adapter size: {total_size / 1e6:.1f} MB")
print(f"Files saved: {os.listdir(adapter_path)}")

# Save logits for ensemble use in Milestone 5
np.save('train_logits_deberta_lora.npy', full_logits)
np.save('test_logits_deberta_lora.npy', test_logits)
print(f"\nLogits saved:")
print(f"  train_logits_deberta_lora.npy — shape {full_logits.shape}")
print(f"  test_logits_deberta_lora.npy  — shape {test_logits.shape}")

LoRA adapter saved to: ./lora_adapter_deberta_v3
Total adapter size: 10.7 MB
Files saved: ['README.md', 'adapter_model.safetensors', 'tokenizer.json', 'tokenizer_config.json', 'adapter_config.json']

Logits saved:
  train_logits_deberta_lora.npy — shape (2000, 5)
  test_logits_deberta_lora.npy  — shape (500, 5)


## 14. Milestone Comparison — The Full Journey So Far

Let's see the progression across all 4 milestones. This table tells the story of our project:

| Milestone | Approach | Key Insight |
|-----------|----------|-------------|
| M1 | TF-IDF / Word2Vec | Surface-level word matching — weak baseline |
| M2 | Sentence-Transformers | Semantic understanding helps, but models are not task-specific |
| M3 | RAG + Cross-Encoder | External context helps some questions, cross-attention is powerful |
| M4 | Fine-tuned DeBERTa | Teaching the model our specific task gives the biggest jump |

In [17]:
comparison = pd.DataFrame({
    'Milestone': ['M1', 'M1', 'M2', 'M3', 'M3', 'M4', 'M4'],
    'Model': [
        'TF-IDF Cosine Similarity',
        'Word2Vec Cosine Similarity',
        'Sentence-Transformer (MiniLM)',
        'Cross-Encoder (no RAG)',
        'Cross-Encoder + RAG (k=3)',
        'DeBERTa-v3 LoRA (val set)',
        'DeBERTa-v3 LoRA (full train)',
    ],
    'MAP@3': [
        0.2962,
        0.3312,
        0.0,    # fill with your actual M2 score
        0.0,    # fill with your actual M3 score
        0.0,    # fill with your actual M3 score
        eval_results['eval_map3'],
        results_ft['map3'],
    ],
})

# NOTE: Replace the 0.0 values above with your actual scores from M2 and M3

display_df = comparison.copy()
display_df['MAP@3'] = display_df['MAP@3'].apply(lambda x: f"{x:.4f}" if x > 0 else "Fill in your score")
print(display_df.to_string(index=False))

# Check against target
print(f"\n{'='*55}")
val_map3 = eval_results['eval_map3']
if val_map3 >= 0.73:
    print(f"  ✓ TARGET CROSSED! Val MAP@3 = {val_map3:.4f} ≥ 0.73")
else:
    print(f"  ✗ Val MAP@3 = {val_map3:.4f} — need {0.73 - val_map3:.4f} more")
    print(f"  → Try the hyperparameter adjustments in the next section")
print(f"{'='*55}")

Milestone                         Model              MAP@3
       M1      TF-IDF Cosine Similarity             0.2962
       M1    Word2Vec Cosine Similarity             0.3312
       M2 Sentence-Transformer (MiniLM) Fill in your score
       M3        Cross-Encoder (no RAG) Fill in your score
       M3     Cross-Encoder + RAG (k=3) Fill in your score
       M4     DeBERTa-v3 LoRA (val set)             0.4794
       M4  DeBERTa-v3 LoRA (full train)             0.4867

  ✗ Val MAP@3 = 0.4794 — need 0.2506 more
  → Try the hyperparameter adjustments in the next section


## 15. Error Analysis — Understanding Model Failures

Error analysis is critical for:
- Your **report** (15 marks) — examiners want to see you understand WHY the model fails
- Your **viva** — "What are the model's limitations?" is a guaranteed question
- **Milestone 5** — knowing failure patterns helps design the ensemble

We analyze three dimensions:
1. **Confidence analysis** — is the model confident when right and uncertain when wrong?
2. **Failure examples** — what do missed questions look like?
3. **Per-answer breakdown** — is the model biased toward certain options?

In [18]:
# ── Per-question scores ────────────────────────────────────────
train_scores = [ap_at_3(t, p) for t, p in zip(train_df['answer'].tolist(), train_pred_labels)]
train_df_analysis = train_df.copy()
train_df_analysis['ap3_score'] = train_scores
train_df_analysis['predicted_top1'] = [p[0] for p in train_pred_labels]
train_df_analysis['predicted'] = [' '.join(p) for p in train_pred_labels]

missed = train_df_analysis[train_df_analysis['ap3_score'] == 0.0]
correct = train_df_analysis[train_df_analysis['ap3_score'] == 1.0]

print(f"Overall Performance:")
print(f"  Perfectly correct (at #1): {len(correct)}/{len(train_df)} ({len(correct)/len(train_df):.1%})")
print(f"  Missed entirely:           {len(missed)}/{len(train_df)} ({len(missed)/len(train_df):.1%})")


# ── Confidence analysis ────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Confidence Analysis")
print(f"{'='*55}")

high_conf_right = 0
low_conf_right = 0
high_conf_wrong = 0
low_conf_wrong = 0

for i in range(len(full_logits)):
    sorted_logits = np.sort(full_logits[i])[::-1]
    margin = sorted_logits[0] - sorted_logits[1]  # gap between top 2 options
    is_correct = train_scores[i] == 1.0
    
    if margin > 2.0:    # high confidence
        if is_correct: high_conf_right += 1
        else: high_conf_wrong += 1
    else:               # low confidence
        if is_correct: low_conf_right += 1
        else: low_conf_wrong += 1

print(f"  High confidence + correct: {high_conf_right}")
print(f"  High confidence + WRONG:   {high_conf_wrong}  ← dangerous overconfidence")
print(f"  Low confidence + correct:  {low_conf_right}")
print(f"  Low confidence + wrong:    {low_conf_wrong}  ← expected uncertainty")


# ── Per-answer breakdown ───────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Accuracy by Answer Label")
print(f"{'='*55}")

for label in ['A', 'B', 'C', 'D', 'E']:
    subset = train_df_analysis[train_df_analysis['answer'] == label]
    correct_subset = subset[subset['ap3_score'] == 1.0]
    acc = len(correct_subset) / len(subset) if len(subset) > 0 else 0
    print(f"  Option {label}: {len(correct_subset)}/{len(subset)} correct ({acc:.1%})")


# ── Failure examples ───────────────────────────────────────────
print(f"\n{'='*55}")
print(f"  Sample Failures (model missed these)")
print(f"{'='*55}")

for i, (_, row) in enumerate(missed.head(3).iterrows()):
    idx = row.name
    print(f"\n  Q{i+1}: {str(row['prompt'])[:130]}...")
    print(f"    Correct:   {row['answer']} = {str(row[row['answer']])[:70]}...")
    print(f"    Predicted: {row['predicted']}")
    logit_str = ", ".join([f"{opt}:{full_logits[idx][j]:.2f}" for j, opt in enumerate(option_cols)])
    print(f"    Logits:    [{logit_str}]")

Overall Performance:
  Perfectly correct (at #1): 631/2000 (31.6%)
  Missed entirely:           557/2000 (27.9%)

  Confidence Analysis
  High confidence + correct: 0
  High confidence + WRONG:   0  ← dangerous overconfidence
  Low confidence + correct:  631
  Low confidence + wrong:    1369  ← expected uncertainty

  Accuracy by Answer Label
  Option A: 92/369 correct (24.9%)
  Option B: 134/490 correct (27.3%)
  Option C: 173/459 correct (37.7%)
  Option D: 112/358 correct (31.3%)
  Option E: 120/324 correct (37.0%)

  Sample Failures (model missed these)

  Q1: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the lis...
    Correct:   B = Martin Heidegger believes that humans do not exist inside time, but th...
    Predicted: D A E
    Logits:    [A:-0.15, B:-0.31, C:-0.30, D:-0.07, E:-0.26]

  Q2: Determine the correct option: What is the term used in astrophysics to describe light-matter interactions resulti

## 16. Finalize W&B & Clean Up

In [19]:
# Log final summary to W&B
wandb.log({
    "final_val_map3": eval_results['eval_map3'],
    "final_val_accuracy": eval_results['eval_accuracy'],
    "final_val_loss": eval_results['eval_loss'],
    "full_train_map3": results_ft['map3'],
    "full_train_top1_acc": results_ft['top1_acc'],
    "full_train_top3_acc": results_ft['top3_acc'],
    "target_crossed": eval_results['eval_map3'] >= 0.73,
})
wandb.finish()
print("W&B run finalized!")

# Clean up GPU memory
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

wandb: updating run metadata
wandb: uploading history steps 68-69, summary, console lines 29-103
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▃▄▄▆▅▆▇███
wandb:               eval/loss █▇▆▅▄▃▂▁▁▁▁
wandb:               eval/map3 ▁▂▂▄▆▆▇▇███
wandb:            eval/runtime ▁▁█▁▇▁█▂▇▁▁
wandb: eval/samples_per_second ██▁█▂█▁▇▂██
wandb:   eval/steps_per_second ██▁█▂█▁▇▂██
wandb:      eval/top3_accuracy ▂▁▁▅▇▇█████
wandb:      final_val_accuracy ▁
wandb:          final_val_loss ▁
wandb:          final_val_map3 ▁
wandb:                     +16 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 0.30333
wandb:               eval/loss 3.15234
wandb:               eval/map3 0.47944
wandb:            eval/runtime 6.8473
wandb: eval/samples_per_second 43.813
wandb:   eval/steps_per_second 5.55
wandb:      eval/top3_accuracy 0.72
wandb:      final_val_accuracy 0.30333
wandb:          final_val_loss 3.15234
wandb:          final_val_map3 0.47944
wandb:                     +22

W&B run finalized!
GPU memory cleared.


In [20]:
# Final submission verification
submission = pd.read_csv('submission.csv')
print(f"Submission shape: {submission.shape}")
print(f"Columns: {list(submission.columns)}")
print(submission.head(10))

assert submission.shape[0] == len(test_df), "Row count mismatch!"
assert all(len(p.split()) == 3 for p in submission['prediction']), "All predictions must have 3 labels!"
print(f"\n✓ All {len(submission)} rows verified. Ready to submit!")

Submission shape: (500, 2)
Columns: ['id', 'prediction']
   id prediction
0   1      B D E
1   2      D B C
2   3      A C D
3   4      C E A
4   5      E D A
5   6      B C D
6   7      D E C
7   8      A E B
8   9      D C E
9  10      E D B

✓ All 500 rows verified. Ready to submit!
